Khushi Khatri BEB222 Experiment 5
Aim:

Theory
Experiment 5: N-Gram Model

Introduction

An N-gram is a contiguous sequence of N items (typically words or characters) extracted from a given piece of text. N-gram models are a foundational technique in statistical Natural Language Processing (NLP) used to capture the local structure and word-order patterns of language. They form the basis of many applications, including language modeling, text prediction (autocomplete), spelling correction, machine translation, speech recognition, and plagiarism detection.

1. Definition

Given a sequence of words w₁, w₂, w₃, ..., wₙ, an N-gram is formed by taking N consecutive words together:

Unigram (N=1): single words — "the," "place," "was," "cozy"
Bigram (N=2): pairs of consecutive words — "the place," "place was," "was cozy"
Trigram (N=3): triplets of consecutive words — "the place was," "place was cozy"
N-gram (general): sequences of N consecutive words

For a sentence with n words, the number of possible N-grams generated is (n − N + 1).

In [1]:
import re
import pandas as pd
from collections import Counter, defaultdict
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.tokenize import word_tokenize
from nltk.util import ngrams

print('Setup complete.')

Setup complete.


[nltk_data] Downloading package punkt to /home/computer/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/computer/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_csv('Balanced_Airbnb_Reviews_Dataset.csv')
print('Shape:', df.shape)
df[['review_id', 'review_text']].head()

Shape: (15000, 42)


,review_id,review_text
0,369314882,Amazing stay! The place felt very cozy for 4 g...
1,490116563,It was okay for the price. Location in XIII Au...
2,582235668,Loved every minute of it. Our superhost was su...
3,68054683,Decent stay overall. Some things could be impr...
4,248483824,Reasonable for a short trip. Location in Long ...


In [3]:
def generate_ngrams(text, n):
    """Generate a list of n-grams from the given text."""
    if not isinstance(text, str) or text.strip() == '':
        return []
    tokens = word_tokenize(text.lower())
    tokens = [tok for tok in tokens if tok.isalpha()]  # keep only alphabetic tokens
    n_grams = list(ngrams(tokens, n))
    return n_grams

# Demo on a sample review
sample_text = df['review_text'].iloc[0]
print('Original Text:\n', sample_text, '\n')

unigrams = generate_ngrams(sample_text, 1)
bigrams = generate_ngrams(sample_text, 2)
trigrams = generate_ngrams(sample_text, 3)

print('Unigrams:\n', unigrams)
print('\nBigrams:\n', bigrams)
print('\nTrigrams:\n', trigrams)

Original Text:
 Amazing stay! The place felt very cozy for 4 guests. Check-in was smooth and the amenities were exactly what we needed. 

Unigrams:
 [('amazing',), ('stay',), ('the',), ('place',), ('felt',), ('very',), ('cozy',), ('for',), ('guests',), ('was',), ('smooth',), ('and',), ('the',), ('amenities',), ('were',), ('exactly',), ('what',), ('we',), ('needed',)]

Bigrams:
 [('amazing', 'stay'), ('stay', 'the'), ('the', 'place'), ('place', 'felt'), ('felt', 'very'), ('very', 'cozy'), ('cozy', 'for'), ('for', 'guests'), ('guests', 'was'), ('was', 'smooth'), ('smooth', 'and'), ('and', 'the'), ('the', 'amenities'), ('amenities', 'were'), ('were', 'exactly'), ('exactly', 'what'), ('what', 'we'), ('we', 'needed')]

Trigrams:
 [('amazing', 'stay', 'the'), ('stay', 'the', 'place'), ('the', 'place', 'felt'), ('place', 'felt', 'very'), ('felt', 'very', 'cozy'), ('very', 'cozy', 'for'), ('cozy', 'for', 'guests'), ('for', 'guests', 'was'), ('guests', 'was', 'smooth'), ('was', 'smooth', 'and')

In [4]:
def ngram_frequency(text, n):
    """Return a frequency count of n-grams in the given text."""
    n_grams = generate_ngrams(text, n)
    return Counter(n_grams)

bigram_freq = ngram_frequency(sample_text, 2)
print('Bigram Frequencies:')
for gram, count in bigram_freq.most_common(10):
    print(gram, '->', count)

Bigram Frequencies:
('amazing', 'stay') -> 1
('stay', 'the') -> 1
('the', 'place') -> 1
('place', 'felt') -> 1
('felt', 'very') -> 1
('very', 'cozy') -> 1
('cozy', 'for') -> 1
('for', 'guests') -> 1
('guests', 'was') -> 1
('was', 'smooth') -> 1


In [5]:
def build_bigram_model(corpus):
    """Build a bigram language model: P(word2 | word1) from a list of texts."""
    bigram_counts = defaultdict(Counter)
    unigram_counts = Counter()

    for text in corpus:
        if not isinstance(text, str):
            continue
        tokens = word_tokenize(text.lower())
        tokens = [tok for tok in tokens if tok.isalpha()]
        unigram_counts.update(tokens)
        for w1, w2 in ngrams(tokens, 2):
            bigram_counts[w1][w2] += 1

    return bigram_counts, unigram_counts

def bigram_probability(w1, w2, bigram_counts, unigram_counts):
    """Compute P(w2 | w1) using Maximum Likelihood Estimation."""
    if unigram_counts[w1] == 0:
        return 0.0
    return bigram_counts[w1][w2] / unigram_counts[w1]

# Build the model using all reviews in the dataset
corpus = df['review_text'].tolist()
bigram_counts, unigram_counts = build_bigram_model(corpus)

print('Vocabulary size:', len(unigram_counts))
print("P('was' | 'it')  =", round(bigram_probability('it', 'was', bigram_counts, unigram_counts), 4))
print("P('great' | 'was') =", round(bigram_probability('was', 'great', bigram_counts, unigram_counts), 4))

Vocabulary size: 532
P('was' | 'it')  = 0.3473
P('great' | 'was') = 0.0


In [6]:
def predict_next_word(word, bigram_counts, top_n=5):
    """Return the top_n most likely next words given the current word."""
    word = word.lower()
    if word not in bigram_counts:
        return []
    return bigram_counts[word].most_common(top_n)

# Demo
for w in ['the', 'very', 'was', 'great']:
    print(f"Next word predictions after '{w}':", predict_next_word(w, bigram_counts))

Next word predictions after 'the': [('entire', 5566), ('host', 3371), ('photos', 2550), ('location', 2544), ('place', 2080)]
Next word predictions after 'very': [('cozy', 2080)]
Next word predictions after 'was': [('not', 2550), ('noisier', 2544), ('difficult', 2485), ('confusing', 2446), ('spotless', 2127)]
Next word predictions after 'great': [('location', 2124)]


In [7]:
def extract_ngrams_row(text, n=2):
    """Return the n-grams as a list of joined strings for a single review."""
    n_grams = generate_ngrams(text, n)
    return [' '.join(gram) for gram in n_grams]

df['bigrams'] = df['review_text'].apply(lambda x: extract_ngrams_row(x, 2))
df['trigrams'] = df['review_text'].apply(lambda x: extract_ngrams_row(x, 3))

df[['review_id', 'review_text', 'bigrams', 'trigrams']].head(10)

,review_id,review_text,bigrams,trigrams
0,369314882,Amazing stay! The place felt very cozy for 4 g...,"[amazing stay, stay the, the place, place felt...","[amazing stay the, stay the place, the place f..."
1,490116563,It was okay for the price. Location in XIII Au...,"[it was, was okay, okay for, for the, the pric...","[it was okay, was okay for, okay for the, for ..."
2,582235668,Loved every minute of it. Our superhost was su...,"[loved every, every minute, minute of, of it, ...","[loved every minute, every minute of, minute o..."
3,68054683,Decent stay overall. Some things could be impr...,"[decent stay, stay overall, overall some, some...","[decent stay overall, stay overall some, overa..."
4,248483824,Reasonable for a short trip. Location in Long ...,"[reasonable for, for a, a short, short trip, t...","[reasonable for a, for a short, a short trip, ..."
5,155617131,Decent stay overall. It served its purpose for...,"[decent stay, stay overall, overall it, it ser...","[decent stay overall, stay overall it, overall..."
6,710244614,"Nothing special, but fine. Our superhost was p...","[nothing special, special but, but fine, fine ...","[nothing special but, special but fine, but fi..."
7,299174484,We had a rough experience. The location in Enc...,"[we had, had a, a rough, rough experience, exp...","[we had a, had a rough, a rough experience, ro..."
8,22136604,Perfect for our trip. Check-in was smooth and ...,"[perfect for, for our, our trip, trip was, was...","[perfect for our, for our trip, our trip was, ..."
9,469473761,Would not recommend. The private room in house...,"[would not, not recommend, recommend the, the ...","[would not recommend, not recommend the, recom..."


In [8]:
all_bigrams = Counter([bg for row in df['bigrams'] for bg in row])
all_trigrams = Counter([tg for row in df['trigrams'] for tg in row])

print('Top 15 Bigrams across dataset:')
for gram, count in all_bigrams.most_common(15):
    print(gram, '->', count)

print('\nTop 15 Trigrams across dataset:')
for gram, count in all_trigrams.most_common(15):
    print(gram, '->', count)

Top 15 Bigrams across dataset:
we had -> 6631
location in -> 6144
the entire -> 5566
entire apartment -> 5522
apartment was -> 4462
we needed -> 4191
our stay -> 4060
than expected -> 3994
the host -> 3371
host was -> 3371
our superhost -> 2684
superhost was -> 2684
room in -> 2558
was not -> 2550
not as -> 2550

Top 15 Trigrams across dataset:
the entire apartment -> 4351
entire apartment was -> 3476
the host was -> 3371
our superhost was -> 2684
was not as -> 2550
not as clean -> 2550
as clean as -> 2550
clean as the -> 2550
as the photos -> 2550
the photos suggested -> 2550
the location in -> 2544
was noisier than -> 2544
noisier than expected -> 2544
was difficult to -> 2485
difficult to reach -> 2485


In [9]:
df[['review_id', 'review_text', 'bigrams', 'trigrams']].to_csv('NGram_Airbnb_Reviews.csv', index=False)
print('Saved to NGram_Airbnb_Reviews.csv')

Saved to NGram_Airbnb_Reviews.csv
